## Tapis and MINT Pipeline
#### 1. Dependencies Installation and Setup Secrets

In [10]:
# Installation of the other standard libraries
%pip install python-dotenv tapipy urllib3

Note: you may need to restart the kernel to use updated packages.


In [11]:
import os
import json
import base64
import urllib3
import requests
from dotenv import load_dotenv
from tapipy.tapis import Tapis

load_dotenv()
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class MintMinimalClient:
    """Lightweight client for interacting with MINT"""
    def __init__(self, base_url, user):
        self.base_url = base_url
        self.user = user

    def post_resource(self, resource_type, data):
        url = f"{self.base_url}/{resource_type}"
        response = requests.post(url, json=data)
        if response.status_code in [200, 201]:
            return response.json()
        else:
            raise Exception(f"Error in MINT ({response.status_code}): {response.text}")

# Mint Initialization
MINT_API_URL = "https://api.models.mint.isi.edu/v1"
mint = MintMinimalClient(MINT_API_URL, os.getenv('TAPIS_USERNAME'))

# Tapis Client
t = Tapis(
    base_url=os.getenv('TAPIS_BASE_URL'),
    username=os.getenv('TAPIS_USERNAME'),
    password=os.getenv('TAPIS_PASSWORD'),
    verify=False
)
t.get_tokens()

print(f"✅ Clients ready. Using Direct API for MINT.")

✅ Clientes listos. Usando API Directa para MINT.


#### 2. Client Initialization (Tapis & MINT)

In [12]:
try:
    # 1. Crear Modelo
    model_data = {
        "label": ["Tapis ETL Framework"],
        "description": ["Framework modular para orquestar pipelines de Tapis v3."],
        "hasModelCategory": ["Data Science"]
    }
    model_res = mint.post_resource("models", model_data)
    model_id = model_res['id']
    print(f"✅ Modelo MINT: {model_id}")

    # 2. Crear Software Version
    software_data = {
        "label": ["v1.0.0-tapis"],
        "hasSoftwareImage": ["ghcr.io/sethsterling22/tapis-workflow-automation/tapis-dummy-app:latest"]
    }
    soft_res = mint.post_resource("softwareversions", software_data)
    soft_id = soft_res['id']

    # 3. Crear Configuración (El Puente)
    tapis_metadata = {
        "pipeline_id": "hello-world-pipeline",
        "group_id": os.getenv('TAPIS_GROUP_ID'),
        "base_url": os.getenv('TAPIS_BASE_URL')
    }
    
    config_data = {
        "label": ["Tapis Pipeline Config"],
        "description": [json.dumps(tapis_metadata)], # Aquí vive el vínculo
        "hasSoftwareImage": ["ghcr.io/sethsterling22/tapis-workflow-automation/tapis-dummy-app:latest"]
    }
    config_res = mint.post_resource("modelconfigurations", config_data)
    CONFIG_ID = config_res['id']
    
    print(f"✅ Configuración vinculada exitosamente. ID: {CONFIG_ID}")

except Exception as e:
    print(f"❌ Error en el registro: {e}")

❌ Error en el registro: Error en MINT (404): {"detail":"Not Found"}


In [ ]:
import time

def trigger_tapis_from_mint(config_url):
    # En un entorno real, MINT pasaría el ID de la configuración seleccionada
    print(f"🔍 Recuperando metadatos de la configuración MINT...")
    
    # Obtener la descripción que contiene los datos de Tapis
    resp = requests.get(config_url)
    config_data = resp.json()
    tapis_info = json.loads(config_data['description'][0])
    
    p_id = tapis_info['pipeline_id']
    g_id = tapis_info['group_id']

    print(f"🚀 Disparando Tapis Pipeline: {p_id}")
    run = t.workflows.runPipeline(group_id=g_id, pipeline_id=p_id)
    run_uuid = run.uuid

    # Loop de monitoreo
    while True:
        status_info = t.workflows.getPipelineRun(group_id=g_id, pipeline_id=p_id, pipeline_run_uuid=run_uuid)
        print(f"[{time.strftime('%H:%M:%S')}] Status: {status_info.status}")
        if status_info.status == 'completed':
            print("✅ Tarea completada. Pipeline ETL listo para el siguiente paso.")
            break
        elif status_info.status in ['failed', 'cancelled']:
            print("❌ El pipeline falló.")
            break
        time.sleep(10)

# Ejecutar (usando el ID obtenido en la celda anterior)
# trigger_tapis_from_mint(CONFIG_ID)

#### 3. Step 1 - Create the Abstract Model

In [ ]:
from model_catalog_python_api import Model

# Definir el modelo
new_model = Model(
    label="Tapis ETL & Analysis Framework",
    description="Framework modular para orquestar pipelines de ETL y análisis en Tapis v3.",
    has_model_category="Data Science"
)

try:
    created_model = m_api.models_post(user=os.getenv('TAPIS_USERNAME'), model=new_model)
    MODEL_ID = created_model.id
    print(f"✅ Modelo creado exitosamente: {MODEL_ID}")
except Exception as e:
    print(f"❌ Error creando el modelo: {e}")

#### 4. Step 2 - Register the Software Version

In [ ]:
from model_catalog_python_api import SoftwareVersion

# Esta versión representa el contenedor que corre en Tapis
soft_version = SoftwareVersion(
    label="v1.0.0-tapis-automation",
    description="Imagen de contenedor para procesamiento de logs alojada en GHCR.",
    has_software_image="ghcr.io/sethsterling22/tapis-workflow-automation/tapis-dummy-app:latest"
)

try:
    created_sv = sv_api.softwareversions_post(user=os.getenv('TAPIS_USERNAME'), software_version=soft_version)
    VERSION_ID = created_sv.id
    print(f"✅ Versión de software registrada: {VERSION_ID}")
except Exception as e:
    print(f"❌ Error al registrar versión: {e}")

#### 5. Step 3 - Create the Configuration (The Tapis Bridge)

In [ ]:
from model_catalog_python_api import ModelConfiguration

# Metadatos que el Bridge leerá para ejecutar t.workflows.runPipeline()
tapis_metadata = {
    "workflow_engine": "Tapis-v3",
    "pipeline_id": "hello-world-pipeline", # El ID de tu pipeline en Tapis
    "group_id": os.getenv('TAPIS_GROUP_ID'),
    "execution_type": "function_with_base64_fix" 
}

# La configuración une el Modelo, el Software y los parámetros de Tapis
model_config = ModelConfiguration(
    label="ETL Prep Pipeline Configuration",
    description=json.dumps(tapis_metadata), # Guardamos el puente como JSON
    has_software_image="ghcr.io/sethsterling22/tapis-workflow-automation/tapis-dummy-app:latest"
)

try:
    created_config = config_api.modelconfigurations_post(
        user=os.getenv('TAPIS_USERNAME'), 
        model_configuration=model_config
    )
    CONFIG_ID = created_config.id
    print(f"✅ Configuración de Tapis vinculada a MINT: {CONFIG_ID}")
except Exception as e:
    print(f"❌ Error en la vinculación: {e}")

#### 6. Define Inputs and Outputs (Data Lineage)

In [ ]:
from model_catalog_python_api import DatasetSpecification

# Definir el input que el pipeline de ETL va a procesar
input_spec = DatasetSpecification(
    label="Raw Network Logs",
    has_format="csv",
    description="Logs crudos provenientes de UniFi/Systems."
)

# Nota: En una implementación completa, asociarías este input_spec 
# al CONFIG_ID usando el endpoint de actualización de configuraciones.
print("✅ Especificaciones de datos preparadas para el orquestador.")

#### 7. The "Trigger" Orchestrator (Tapis Status Monitor)

In [ ]:
import time

def run_tapis_step_from_mint(config_id):
    """
    Simula la acción del MINT Execution Engine:
    1. Lee metadatos de MINT.
    2. Ejecuta en Tapis.
    3. Monitorea hasta el éxito.
    """
    # 1. Obtener configuración de MINT
    # El ID suele ser una URL, extraemos la parte final
    c_id = config_id.split('/')[-1]
    config = config_api.modelconfigurations_id_get(id=c_id)
    
    # 2. Parsear metadatos de Tapis guardados en la descripción
    metadata = json.loads(config.description)
    p_id = metadata['pipeline_id']
    g_id = metadata['group_id']
    
    print(f"📡 MINT Trigger: Iniciando Pipeline '{p_id}' en Tapis...")

    # 3. Ejecutar en Tapis
    run = t.workflows.runPipeline(group_id=g_id, pipeline_id=p_id)
    run_uuid = run.uuid
    print(f"🚀 Tapis Run iniciado: {run_uuid}")

    # 4. Monitoreo (Poll Loop)
    while True:
        status_info = t.workflows.getPipelineRun(
            group_id=g_id, 
            pipeline_id=p_id, 
            pipeline_run_uuid=run_uuid
        )
        status = status_info.status
        print(f"[{time.strftime('%H:%M:%S')}] Estado en Tapis: {status}")
        
        if status == 'completed':
            print(f"✅ Tarea finalizada. MINT puede proceder al siguiente modelo en el DAG.")
            return True
        elif status in ['failed', 'cancelled']:
            print(f"❌ Error en Tapis. Deteniendo orquestación de MINT.")
            return False
        
        time.sleep(10)

# Ejemplo de uso (Descomenta cuando CONFIG_ID esté listo):
# run_tapis_step_from_mint(CONFIG_ID)